In [ ]:
import pandas as pd
import re

# ===== 1) Load =====
# change path if needed
df = pd.read_csv("actions.csv")

# Expecting these columns exist (based on your file):
# ['case_id','title','action_no','action','owner','timing','verification']

LABEL_PATTERNS = {
    "owner": r"Owner(?:\s*\(role\))?\s*:\s*",
    "timing": r"Timing\s*:\s*",
    "verification": r"Verification\s*:\s*",
    "action": r"Action\s*:\s*",
}

def _strip_list_prefix(s: str) -> str:
    # remove leading list markers like "a.", "b)", "-", "–", "•"
    return re.sub(r"^\s*(?:[a-zA-Z]\s*[\.\)]|[\-\–\—•])\s*", "", s).strip()

def _clean_spaces(s: str) -> str:
    s = "" if pd.isna(s) else str(s)
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _extract_labeled(blob: str, label: str) -> str | None:
    """
    Robustly extract labeled fields from a messy multi-line blob, handling:
    - Owner vs Owner(role)
    - inline spillovers like: "Owner: X\nb. Timing: Y\nc. Verification: Z"
    - different dividers / bullets / lettered subitems
    """
    if not blob:
        return None

    if label == "owner":
        pat = LABEL_PATTERNS["owner"] + r"(.*?)(?=\n\s*(?:[a-zA-Z]\s*[\.\)]\s*)?(?:Timing|Verification)\s*:|\Z)"
    elif label == "timing":
        pat = LABEL_PATTERNS["timing"] + r"(.*?)(?=\n\s*(?:[a-zA-Z]\s*[\.\)]\s*)?(?:Owner|Verification)\s*:|\Z)"
    elif label == "verification":
        pat = LABEL_PATTERNS["verification"] + r"(.*?)(?=\n\s*(?:[a-zA-Z]\s*[\.\)]\s*)?(?:Owner|Timing)\s*:|\Z)"
    elif label == "action":
        pat = LABEL_PATTERNS["action"] + r"(.*?)(?=\n\s*(?:[a-zA-Z]\s*[\.\)]\s*)?(?:Owner|Timing|Verification)\s*:|\Z)"
    else:
        return None

    m = re.search(pat, blob, flags=re.IGNORECASE | re.DOTALL)
    if not m:
        return None

    val = m.group(1).strip()

    # if any label accidentally appears inline, cut it off
    val = re.split(r"\b(?:Owner(?:\s*\(role\))?|Timing|Verification)\s*:\s*", val, flags=re.IGNORECASE)[0].strip()

    val = _strip_list_prefix(val)
    val = re.sub(r"\s+", " ", val).strip()
    return val or None

def normalize_row(row: pd.Series) -> pd.Series:
    action = str(row.get("action", ""))
    owner = str(row.get("owner", ""))
    timing = str(row.get("timing", ""))
    verification = str(row.get("verification", ""))

    # Build a single blob so regex can “recover” when fields spill into each other
    blob = "\n".join([
        f"Action: {action}",
        f"Owner: {owner}",
        f"Timing: {timing}",
        f"Verification: {verification}",
    ])

    action_title = _extract_labeled(blob, "action") or _clean_spaces(action)
    owner_out    = _extract_labeled(blob, "owner") or _clean_spaces(owner)
    timing_out   = _extract_labeled(blob, "timing") or _clean_spaces(timing)
    verif_out    = _extract_labeled(blob, "verification") or _clean_spaces(verification)

    # final safety cleanup: remove any leftover "Timing:" etc that might remain
    for lbl in ["Owner", "Owner (role)", "Timing", "Verification", "Action"]:
        owner_out  = re.sub(rf"\b{re.escape(lbl)}\s*:\s*", "", owner_out,  flags=re.IGNORECASE).strip()
        timing_out = re.sub(rf"\b{re.escape(lbl)}\s*:\s*", "", timing_out, flags=re.IGNORECASE).strip()
        verif_out  = re.sub(rf"\b{re.escape(lbl)}\s*:\s*", "", verif_out,  flags=re.IGNORECASE).strip()
        action_title = re.sub(rf"\b{re.escape(lbl)}\s*:\s*", "", action_title, flags=re.IGNORECASE).strip()

    return pd.Series({
        "case_id": row["case_id"],
        "case_title": row["title"],
        "action_id": row["action_no"],
        "action_title": action_title,
        "owner": owner_out,
        "timing": timing_out,
        "verification": verif_out,
    })

# ===== 2) Clean =====
cleaned = df.apply(normalize_row, axis=1)

# (optional) quick sanity checks: should be 0
for col in ["owner", "timing", "verification", "action_title"]:
    bad = cleaned[col].astype(str).str.contains(r"(Owner|Timing|Verification)\s*:", case=False, na=False).sum()
    print(col, "rows still containing labels:", bad)

# ===== 3) Save =====
cleaned.to_csv("actions_cleaned.csv", index=False)

print("Saved:", "actions_cleaned.csv")
print(cleaned.head())

owner rows still containing labels: 0
timing rows still containing labels: 0
verification rows still containing labels: 0
action_title rows still containing labels: 0
Saved: actions_cleaned.csv
   case_id                                         case_title  action_id  \
0        1  Minor Vapor Release During Instrument Tubing R...          1   
1        1  Minor Vapor Release During Instrument Tubing R...          2   
2        1  Minor Vapor Release During Instrument Tubing R...          3   
3        1  Minor Vapor Release During Instrument Tubing R...          4   
4        1  Minor Vapor Release During Instrument Tubing R...          5   

                                        action_title  \
0  Add explicit vent-point check to all transmitt...   
1  Install high-visibility tags on local instrume...   
2  Update isolation standard to include guidance ...   
3  Conduct refresher training on identifying pres...   
4  Implement mandatory “vent-verified” checkbox o...   

            

/tmp/ipython-input-27068146.py:103: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  bad = cleaned[col].astype(str).str.contains(r"(Owner|Timing|Verification)\s*:", case=False, na=False).sum()
/tmp/ipython-input-27068146.py:103: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  bad = cleaned[col].astype(str).str.contains(r"(Owner|Timing|Verification)\s*:", case=False, na=False).sum()
/tmp/ipython-input-27068146.py:103: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  bad = cleaned[col].astype(str).str.contains(r"(Owner|Timing|Verification)\s*:", case=False, na=False).sum()
/tmp/ipython-input-27068146.py:103: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.ext

# EDA check

In [ ]:
import pandas as pd

df = pd.read_csv("actions_cleaned.csv")

df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1603 entries, 0 to 1602
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   case_id       1603 non-null   int64 
 1   case_title    1603 non-null   object
 2   action_id     1603 non-null   int64 
 3   action_title  1603 non-null   object
 4   owner         1603 non-null   object
 5   timing        1603 non-null   object
 6   verification  1603 non-null   object
dtypes: int64(2), object(5)
memory usage: 87.8+ KB


,case_id,case_title,action_id,action_title,owner,timing,verification
0,1,Minor Vapor Release During Instrument Tubing R...,1,Add explicit vent-point check to all transmitt...,Maintenance Planner,<30 days,Spot audit of next 10 relevant jobs.
1,1,Minor Vapor Release During Instrument Tubing R...,2,Install high-visibility tags on local instrume...,Operations Supervisor,30–90 days,Field walkdown signoff.
2,1,Minor Vapor Release During Instrument Tubing R...,3,Update isolation standard to include guidance ...,Process Safety Engineer,>90 days,Procedure approval plus training record review.
3,1,Minor Vapor Release During Instrument Tubing R...,4,Conduct refresher training on identifying pres...,HSE Advisor,<30 days,Attendance log & post-training quiz.
4,1,Minor Vapor Release During Instrument Tubing R...,5,Implement mandatory “vent-verified” checkbox o...,Permit to Work Coordinator,Immediate,Weekly PTW compliance review.


 ## Identify categorical columns

In [ ]:
cat_cols = ["owner", "timing", "case_title"]

df[cat_cols].describe()

,owner,timing,case_title
count,1603,1603,1603
unique,413,16,191
top,Operations Supervisor,30–90 days,Arc Flash During Overhead Panel Removal
freq,70,387,20


In [ ]:
# missing values check
df[cat_cols].isna().sum().sort_values(ascending=False)

,0
owner,0
timing,0
case_title,0


## Category counts

In [ ]:
df["owner"].value_counts()

,count
owner,
Operations Supervisor,70
HSE Advisor,61
Reliability Engineer,48
Internal Audit Analyst,36
Control Systems Engineer,27
...,...
Maintenance Planning Coordinator,1
Facilities & Maintenance Lead,1
Operations & HSE Lead,1


In [ ]:
df["timing"].value_counts()

,count
timing,
30–90 days,387
<30 days,350
Immediate,292
>90 days,279
<30 days ●,83
30–90 days ●,81
>90 days ●,67
Immediate ●,51
<30 days o,3


In [ ]:
df["case_title"].value_counts()

,count
case_title,
Arc Flash During Overhead Panel Removal,20
Home Visitor Overhears Confidential,14
Unexpected Pressure Release During Sampling Line Replacement,10
Solvent Transfer Splash During Sample Prep,10
Solvent Bottle Tipped During Waste Consolidation,10
...,...
Laptop Theft in Public Café During Remote Work,5
Sensitive Project Discussion Overheard During Café Meeting,5
Misconfigured Sensor Mapping in Predictive Model Deployment,5


# Further Cleaning

## timing

In [ ]:
# Clean weird symbols & annotations
import re

def clean_timing(text):
    if pd.isna(text):
        return text

    text = str(text)

    # remove annotations in parentheses
    text = re.sub(r"\(.*?\)", "", text)

    # remove bullets, dots, symbols
    text = re.sub(r"[•●▪◦·*o]+", "", text)

    # remove extra punctuation
    text = re.sub(r"[^\w\s<>–-]", "", text)

    # normalize dash variations
    text = text.replace("-", "–")

    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# standardize categories
def normalize_timing(text):
    if pd.isna(text):
        return text

    t = text.lower()

    if "immediate" in t:
        return "Immediate"

    if "<30" in t:
        return "<30 days"

    if "30–90" in t or "30 90" in t:
        return "30–90 days"

    if ">90" in t or "90+" in t:
        return ">90 days"

    # group all mid-range durations
    if "<60" in t or "<90" in t or "30–90" in t or "30-90" in t:
        return "30–90 days"


    return text  # fallback for unexpected cases

In [ ]:
# apply cleaning
df["timing"] = df["timing"].apply(clean_timing)
df["timing"] = df["timing"].apply(normalize_timing)

In [ ]:
df["timing"].value_counts()

,count
timing,
30–90 days,473
<30 days,437
>90 days,348
Immediate,345


## owner

In [ ]:
import pandas as pd
import re

# --- 1) basic normalization to remove weird bullets/dividers + unify spacing ---
def clean_owner_raw(x):
    if pd.isna(x):
        return ""

    s = str(x)

    # drop labels if they accidentally leaked in
    s = re.sub(r"(?i)\bOwner(?:\s*\(role\))?\s*:\s*", "", s)

    # remove bullets / odd dividers / repeated punctuation
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[•●▪◦·*]+", " ", s)
    s = re.sub(r"[|/\\]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s

# --- 2) bucketing rules (priority-ordered) ---
# Important: order matters. We'll apply rules top-to-bottom.
OWNER_BUCKET_RULES = [
    # --- IT/Digital first: lots of roles are admin/security but not “it/digital/data/analytics” ---
    ("IT/Digital",
     r"\b("
     r"it|digital|data|analytics|ai|"
     r"information(\s*management|\s*systems)?|"
     r"system(s)?|platform|application|app|software|"
     r"administrator|admin|architect|engineer|developer|specialist|analyst|"
     r"configuration|service\s*management|helpdesk|support|"
     r"network|endpoint|device|mobile|monitoring|messaging|collaboration|"
     r"iam|identity|access\s*management|access\s*control|"
     r"ot\s*security|cyber(security)?|security\s*(governance|compliance|awareness)"
     r")\b"),

    # --- HSE: broaden beyond literal "hse/safety" ---
    ("HSE",
     r"\b("
     r"hse|hsseq|hsse|safety|hazard|risk|"
     r"environment(al)?|occupational\s*health|health|"
     r"emergency\s*response|incident\s*response|"
     r"security(\s*systems)?(\s*technician)?|"
     r"gas\s*tester|"
     r"audit|assurance|compliance"
     r")\b"),

    # --- Engineering/Reliability: cover instrumentation/automation/electrical/mechanical integrity ---
    ("Engineering/Reliability",
     r"\b("
     r"engineer(ing)?|reliability|process\s*safety|"
     r"instrumentation|automation|electrical|commissioning|"
     r"mechanical\s*integrity|integrity|moc|management\s*of\s*change|"
     r"technical"
     r"hmi|ux"
     r")\b"),

    # --- Maintenance: broaden beyond maintenance/planner/mechanic ---
    ("Maintenance",
     r"\b("
     r"maintenance|planner|mechanic|"
     r"mechanical\b|"
     r"cmms|work\s*order|workpack|"
     r"scaffolding|"
     r"shop|workshop|craft"
     r")\b"),

    # --- Operations: broaden beyond operations/ops/shift supervisor ---
    ("Operations",
     r"\b("
     r"operations?|ops|shift|control\s*room|"
     r"performing\s*authority|\bpa\b|"
     r"area\s*authority|"
     r"ptw|permit(\s*to\s*work)?|permit\s*(issuer|coordinator)|"
     r"simops"
     r")\b"),

    # --- Facilities/Field: broaden facilities/field/workplace/office services ---
    ("Facilities/Field",
     r"\b("
    r"facilities?|field|workplace|office\s*services|reception|"
     r"warehouse|logistics|stores?|procurement|purchasing|"
     r"site\s*scaffolding"
     r")\b"),

    # --- Contractor Mgmt: broaden vendor/contractor wording ---
    ("Contractor Mgmt",
    r"\b(contractor|vendor|third\s*party)\b"
    ),

    # --- HR/Learning: broaden beyond hr/human resources/learning/training ---
    ("HR/Learning",
     r"\b("
      r"hr|human\s*resources|people(\s*&\s*culture)?|"
     r"learning|training|onboarding|"
     r"hris"
     r")\b"),

    # --- Management: broaden beyond manager/director/superintendent ---
    ("Management",
     r"\b("
     r"manager|managers|director|superintendent|"
     r"supervisor|supervisors|lead|head|coordinator|"
     r"corporate\s*communications"
     r")\b"),
]


def bucket_owner(owner):
    s = clean_owner_raw(owner)
    s_l = s.lower().strip()

    if s_l == "":
        return "Unknown"

    for bucket, pat in OWNER_BUCKET_RULES:
        if re.search(pat, s_l, flags=re.IGNORECASE):
            return bucket

    return "Other"

# --- 3) apply ---
# assumes your cleaned dataset column is named exactly "owner"
df["owner_clean"] = df["owner"].apply(clean_owner_raw)
df["owner_bucket"] = df["owner_clean"].apply(bucket_owner)

# --- 4) quick checks ---
print("Top raw owners:")
print(df["owner"].value_counts().head(15), "\n")

print("Bucket counts:")
print(df["owner_bucket"].value_counts(), "\n")

# See examples per bucket (helps validate rules)
for b in df["owner_bucket"].value_counts().index:
    examples = (df.loc[df["owner_bucket"] == b, "owner_clean"]
                  .dropna()
                  .astype(str)
                  .value_counts()
                  .head(8))
    print(f"\n== {b} (examples) ==")
    print(examples)

# --- 5) (optional) overwrite owner with bucket for downstream analysis ---
#df["owner"] = df["owner_bucket"]

df["owner"] = df["owner_clean"]
df.drop(columns=["owner_clean"], inplace=True)


Top raw owners:
owner
Operations Supervisor                 88
HSE Advisor                           83
Reliability Engineer                  69
Training Coordinator                  41
Internal Audit Analyst                36
Area Authority                        27
Control Systems Engineer              27
HSE Training Advisor                  27
Operations Manager                    26
AI Platform Engineer                  25
Digital Tools Training Coordinator    24
Maintenance Lead                      24
Systems Configuration Specialist      23
IT Security Lead                      23
HSE Training Coordinator              22
Name: count, dtype: int64 

Bucket counts:
owner_bucket
IT/Digital                 680
HSE                        258
Operations                 225
Engineering/Reliability    128
Maintenance                 90
Management                  71
HR/Learning                 63
Facilities/Field            57
Contractor Mgmt             31
Name: count, dtype: int64 



In [ ]:
df["owner_bucket"].value_counts()

,count
owner_bucket,
IT/Digital,680
HSE,258
Operations,225
Engineering/Reliability,128
Maintenance,90
Management,71
HR/Learning,63
Facilities/Field,57
Contractor Mgmt,31


In [ ]:
df["owner"].value_counts()

,count
owner,
Operations Supervisor,88
HSE Advisor,83
Reliability Engineer,69
Training Coordinator,41
Internal Audit Analyst,36
...,...
Digital Systems Lead,1
Maintenance Planning Coordinator,1
Mechanical PA,1


In [ ]:
df.head(5)

,case_id,case_title,action_id,action_title,owner,timing,verification,owner_bucket
0,1,Minor Vapor Release During Instrument Tubing R...,1,Add explicit vent-point check to all transmitt...,Maintenance Planner,<30 days,Spot audit of next 10 relevant jobs.,Maintenance
1,1,Minor Vapor Release During Instrument Tubing R...,2,Install high-visibility tags on local instrume...,Operations Supervisor,30–90 days,Field walkdown signoff.,Operations
2,1,Minor Vapor Release During Instrument Tubing R...,3,Update isolation standard to include guidance ...,Process Safety Engineer,>90 days,Procedure approval plus training record review.,HSE
3,1,Minor Vapor Release During Instrument Tubing R...,4,Conduct refresher training on identifying pres...,HSE Advisor,<30 days,Attendance log & post-training quiz.,HSE
4,1,Minor Vapor Release During Instrument Tubing R...,5,Implement mandatory “vent-verified” checkbox o...,Permit to Work Coordinator,Immediate,Weekly PTW compliance review.,Other


In [ ]:
# save
df.to_csv("actions_cleaned_v2.csv", index=False)
print("\nSaved: actions_cleaned_v2.csv")


Saved: actions_cleaned_v2.csv


# Add label features

## Classify actions into action types

In [ ]:
def _norm_text(x):
    """Lowercase, collapse spaces, keep it safe for regex search."""
    if pd.isna(x):
        return ""
    s = str(x).lower()
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [ ]:
# ==========================================================
# 1) Action Type classification (first matching category)
# ==========================================================

ACTION_TYPE_RULES = [
    ("Procedure/Standard", r"\b(procedure|sop|standard|checklist|permit|policy)\b"),
    ("Training/Drills", r"\b(training|refresher|toolbox|drill|awareness|quiz)\b"),
    ("Equipment/Engineering Change", r"\b(install|add|modify|upgrade|engineering|equipment|hardware)\b"),
    ("Visual Management/Signage", r"\b(label|signage|tag|marker|visual)\b"),
    ("Communication/Coordination", r"\b(handover|hand\-over|communication|radio|coordination|protocol)\b"),
    ("Audit/Verification/Assessment", r"\b(audit|inspection|inspect|verify|review|assessment|walkdown|walk\-down)\b"),
]

def classify_action_type(action_title):
    t = _norm_text(action_title)

    if t == "":
        return "Unknown"

    for cat, pat in ACTION_TYPE_RULES:
        if re.search(pat, t, flags=re.IGNORECASE):
            return cat

    return "Other"

df["action_type"] = df["action_title"].apply(classify_action_type)

df["action_type"].value_counts()

,count
action_type,
Other,808
Equipment/Engineering Change,370
Training/Drills,178
Procedure/Standard,91
Audit/Verification/Assessment,83
Communication/Coordination,47
Visual Management/Signage,26


## Map each action type to a control level (Engineering vs Administrative).

In [ ]:
# ==========================================================
# 2) Map action type -> control level
# ==========================================================

CONTROL_LEVEL_MAP = {
    "Equipment/Engineering Change": "Engineering Control",
    "Procedure/Standard": "Administrative Control",
    "Training/Drills": "Administrative Control",
    "Communication/Coordination": "Administrative Control",
    "Audit/Verification/Assessment": "Administrative Control",
    "Visual Management/Signage": "Hybrid",
    "Other": "Administrative Control",   # choose your preference
    "Unknown": "Unknown",
}

df["control_level"] = df["action_type"].map(CONTROL_LEVEL_MAP).fillna("Unknown")

df["control_level"].value_counts()

,count
control_level,
Administrative Control,1207
Engineering Control,370
Hybrid,26


## Derive verification-strength features

by detecting whether audits/logs/tests were mentioned. (strength = How objectively completion is checked)

+1 if audit/inspection mentioned

+1 if logs/records mentioned

+1 if testing/validation mentioned

In [ ]:
# ==========================================================
# 3) Verification-strength features (audit/logs/testing)
# ==========================================================

AUDIT_PAT = r"\b(audit|inspection|inspect)\b"
LOGS_PAT = r"\b(log|logs|record|records|recordkeeping|cmms)\b"
TEST_PAT = r"\b(test|testing|validated|validation|verify|verification|signoff|sign\-off|walkdown|walk\-down)\b"

def verification_strength(verification_text):
    v = _norm_text(verification_text)

    score = 0
    audit_flag = 1 if re.search(AUDIT_PAT, v) else 0
    logs_flag  = 1 if re.search(LOGS_PAT, v) else 0
    test_flag  = 1 if re.search(TEST_PAT, v) else 0

    score = audit_flag + logs_flag + test_flag
    return pd.Series({
        "verif_has_audit": audit_flag,
        "verif_has_logs_records": logs_flag,
        "verif_has_testing_validation": test_flag,
        "verification_strength": score
    })

df[[
    "verif_has_audit",
    "verif_has_logs_records",
    "verif_has_testing_validation",
    "verification_strength"
]] = df["verification"].apply(verification_strength)

df["verification_strength"].value_counts().sort_index()

,count
verification_strength,
0,712
1,812
2,76
3,3


In [ ]:
df.drop(columns=["verif_has_audit", "verif_has_logs_records", "verif_has_testing_validation"])

,case_id,case_title,action_id,action_title,owner,timing,verification,owner_bucket,action_type,control_level,verification_strength,timing_score,is_short_term,is_long_term
0,1,Minor Vapor Release During Instrument Tubing R...,1,Add explicit vent-point check to all transmitt...,Maintenance Planner,<30 days,Spot audit of next 10 relevant jobs.,Maintenance,Equipment/Engineering Change,Engineering Control,1,1,1,0
1,1,Minor Vapor Release During Instrument Tubing R...,2,Install high-visibility tags on local instrume...,Operations Supervisor,30–90 days,Field walkdown signoff.,Operations,Equipment/Engineering Change,Engineering Control,1,2,0,0
2,1,Minor Vapor Release During Instrument Tubing R...,3,Update isolation standard to include guidance ...,Process Safety Engineer,>90 days,Procedure approval plus training record review.,HSE,Procedure/Standard,Administrative Control,1,3,0,1
3,1,Minor Vapor Release During Instrument Tubing R...,4,Conduct refresher training on identifying pres...,HSE Advisor,<30 days,Attendance log & post-training quiz.,HSE,Training/Drills,Administrative Control,1,1,1,0
4,1,Minor Vapor Release During Instrument Tubing R...,5,Implement mandatory “vent-verified” checkbox o...,Permit to Work Coordinator,Immediate,Weekly PTW compliance review.,Other,Procedure/Standard,Administrative Control,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1598,196,Mis‑Mapped Tie‑In Boundary Led to Incorrect,5,Implement secondary pressure verification at m...,Operations Lead,<30 days,PTW records show readings from two or more gau...,Operations,Other,Administrative Control,1,1,1,0
1599,196,Mis‑Mapped Tie‑In Boundary Led to Incorrect,6,Add a training module on complex/legacy piping...,HSE Training Advisor,<30 days,LMS completion reports.,HSE,Training/Drills,Administrative Control,0,1,1,0
1600,196,Mis‑Mapped Tie‑In Boundary Led to Incorrect,7,Create an MOC checklist item requiring isolati...,Engineering Manager,>90 days,MOC audits reflect consistent use.,Engineering/Reliability,Procedure/Standard,Administrative Control,0,3,0,1
1601,196,Mis‑Mapped Tie‑In Boundary Led to Incorrect,8,Establish routine inspections of legacy valves...,Maintenance Planner,30–90 days,PM tasks created; inspection reports logged.,Maintenance,Other,Administrative Control,1,2,0,0


## Flags for short- vs long-term actions

In [ ]:
# ==========================================================
# 4) Short vs long term flags (via timing_score)
# ==========================================================
# timing_score definition:
# 0 = Immediate
# 1 = <30 days
# 2 = 30–90 days
# 3 = >90 days

TIMING_SCORE_MAP = {
    "Immediate": 0,
    "<30 days": 1,
    "30–90 days": 2,
    ">90 days": 3,
}

df["timing_score"] = df["timing"].map(TIMING_SCORE_MAP)

df["is_short_term"] = df["timing_score"].isin([0, 1]).astype(int)
df["is_long_term"]  = (df["timing_score"] == 3).astype(int)

df[["timing", "timing_score", "is_short_term", "is_long_term"]].head(10)

,timing,timing_score,is_short_term,is_long_term
0,<30 days,1,1,0
1,30–90 days,2,0,0
2,>90 days,3,0,1
3,<30 days,1,1,0
4,Immediate,0,1,0
5,30–90 days,2,0,0
6,Immediate,0,1,0
7,>90 days,3,0,1
8,Immediate,0,1,0
9,<30 days,1,1,0


In [ ]:
df["timing_score"].value_counts().sort_index()

,count
timing_score,
0,345
1,437
2,473
3,348


In [ ]:
df.head(5)

,case_id,case_title,action_id,action_title,owner,timing,verification,owner_bucket,action_type,control_level,verif_has_audit,verif_has_logs_records,verif_has_testing_validation,verification_strength,timing_score,is_short_term,is_long_term
0,1,Minor Vapor Release During Instrument Tubing R...,1,Add explicit vent-point check to all transmitt...,Maintenance Planner,<30 days,Spot audit of next 10 relevant jobs.,Maintenance,Equipment/Engineering Change,Engineering Control,1,0,0,1,1,1,0
1,1,Minor Vapor Release During Instrument Tubing R...,2,Install high-visibility tags on local instrume...,Operations Supervisor,30–90 days,Field walkdown signoff.,Operations,Equipment/Engineering Change,Engineering Control,0,0,1,1,2,0,0
2,1,Minor Vapor Release During Instrument Tubing R...,3,Update isolation standard to include guidance ...,Process Safety Engineer,>90 days,Procedure approval plus training record review.,HSE,Procedure/Standard,Administrative Control,0,1,0,1,3,0,1
3,1,Minor Vapor Release During Instrument Tubing R...,4,Conduct refresher training on identifying pres...,HSE Advisor,<30 days,Attendance log & post-training quiz.,HSE,Training/Drills,Administrative Control,0,1,0,1,1,1,0
4,1,Minor Vapor Release During Instrument Tubing R...,5,Implement mandatory “vent-verified” checkbox o...,Permit to Work Coordinator,Immediate,Weekly PTW compliance review.,Other,Procedure/Standard,Administrative Control,0,0,0,0,0,1,0


In [ ]:
# Save
df.to_csv("actions_cleaned_v3.csv", index=False)
print("Saved: actions_cleaned_v3.csv")

Saved: actions_cleaned_v3.csv


# Feature Engineering

In [ ]:
import pandas as pd

df = pd.read_csv("actions_cleaned_v3.csv")


In [ ]:
df.loc[df["owner_bucket"] == "Other", "owner"].dropna().unique()

array([], dtype=object)

In [ ]:
# Save
df.to_csv("actions_cleaned_v4.csv", index=False)
print("Saved: actions_cleaned_v4.csv")

Saved: actions_cleaned_v4.csv


In [ ]:
df.head(5)

,case_id,case_title,action_id,action_title,owner,timing,verification,owner_bucket,action_type,control_level,verif_has_audit,verif_has_logs_records,verif_has_testing_validation,verification_strength,timing_score,is_short_term,is_long_term
0,1,Minor Vapor Release During Instrument Tubing R...,1,Add explicit vent-point check to all transmitt...,Maintenance Planner,<30 days,Spot audit of next 10 relevant jobs.,Maintenance,Equipment/Engineering Change,Engineering Control,1,0,0,1,1,1,0
1,1,Minor Vapor Release During Instrument Tubing R...,2,Install high-visibility tags on local instrume...,Operations Supervisor,30–90 days,Field walkdown signoff.,Operations,Equipment/Engineering Change,Engineering Control,0,0,1,1,2,0,0
2,1,Minor Vapor Release During Instrument Tubing R...,3,Update isolation standard to include guidance ...,Process Safety Engineer,>90 days,Procedure approval plus training record review.,IT/Digital,Procedure/Standard,Administrative Control,0,1,0,1,3,0,1
3,1,Minor Vapor Release During Instrument Tubing R...,4,Conduct refresher training on identifying pres...,HSE Advisor,<30 days,Attendance log & post-training quiz.,HSE,Training/Drills,Administrative Control,0,1,0,1,1,1,0
4,1,Minor Vapor Release During Instrument Tubing R...,5,Implement mandatory “vent-verified” checkbox o...,Permit to Work Coordinator,Immediate,Weekly PTW compliance review.,Operations,Procedure/Standard,Administrative Control,0,0,0,0,0,1,0


## Action_type

In [ ]:
df["action_type"].value_counts()

,count
action_type,
Audit/Verification/Assessment,504
Equipment/Engineering Change,323
Other,247
Procedure/Standard,235
Training/Drills,175
Visual Management/Signage,102
Communication/Coordination,17


In [ ]:
# what action titles are classified as Other?
df.loc[df["action_type"] == "Other", "action_title"].value_counts().head(30)

,count
action_title,
Procure size-specific dispenser tips to ensure proper fit with volumetric flasks.,1
"Evaluate alternative dispensing methods (pipette-pump, dropper bottle) for high-risk solvents.",1
Procure fitted lab-coat sleeves or sleeve-retainers for solvent-handling tasks.,1
Rearrange collaborative spaces to provide wider movement paths.,1
Provide cable-management solutions that elevate or protect power strips from spills.,1
Deliver a short safety reminder during next all-hands meeting on safe transport of liquids.,1
Introduce routine morning floor sweeps for congestion-prone areas.,1
Provide spill kits in office kitchenette areas.,1
Encourage team leads to relocate discussions to meeting rooms during peak traffic periods.,1


In [ ]:
# =========================
# 0) keep your normalizer
# =========================
import re
import pandas as pd

def _norm_text(x):
    """Lowercase, collapse spaces, keep it safe for regex search."""
    if pd.isna(x):
        return ""
    s = str(x).lower()
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [ ]:
# ==========================================================
# Replace your ACTION_TYPE_RULES with this richer word bank
# (still ONLY your 6 categories; still "first match wins")
# Goal: pull many current "Other" titles into existing buckets
# based on what we see in your "Other" list (P&IDs, walkdowns,
# audits, checks, layouts, tracking systems, housekeeping, etc.)
# ==========================================================

ACTION_TYPE_RULES = [
    # 1) Audit / Verification / Assessment  (put FIRST to capture "conduct audits/walkdowns/checks")
    ("Audit/Verification/Assessment",
     r"\b("
     r"audit(s)?|auditing|"
     r"inspection(s)?|inspect(ing)?|"
     r"assessment(s)?|assess(ing)?|"
     r"review(s)?|review(ing)?|"
     r"verify|verification|validated|validation|"
     r"test(s)?|testing|"
     r"walkdown(s)?|walk[- ]down(s)?|field\s*walkdown(s)?|field\s*check(s)?|"
     r"spot\s*check(s)?|"
     r"sign[- ]off|approval|"
     r"confirm(ation)?|"
     r"accuracy|drift"
     r")\b"),

    # 2) Visual Management / Signage  (capture tags/labels/markers/visibility/color-coded/QR)
    ("Visual Management/Signage",
     r"\b("
     r"label(l)?(ing)?|"
     r"signage|sign(s)?|"
     r"tag(s)?|tagging|"
     r"marker(s)?|"
     r"high[- ]visibility|visibility|"
     r"color[- ]?coded?|lanyard(s)?|"
     r"qr[- ]?code(d)?|"
     r"plate(s)?"
     r")\b"),

    # 3) Procedure / Standard  (capture SOPs + "rules/guidelines/requirements/templates/forms/LOTO")
    ("Procedure/Standard",
     r"\b("
     r"procedure(s)?|sop(s)?|standard(s)?|"
     r"checklist(s)?|permit(s)?|policy|"
     r"loto|lockout|tagout|"
     r"rule(s)?|guideline(s)?|"
     r"template(s)?|form(s)?|"
     r"protocol(s)?|"
     r"require(s|d)?|mandatory|"
     r"hierarchy|"
     r"documentation\s*gap(s)?|documentation\s*accuracy|"
     r"document\s*control|"
     r"single[- ]source[- ]of[- ]truth"
     r")\b"),

    # 4) Training / Drills  (toolbox, briefings, campaigns, awareness)
    ("Training/Drills",
     r"\b("
     r"training|refresher|toolbox|toolbox\s*talk|"
     r"drill(s)?|exercise(s)?|"
     r"awareness|quiz|"
     r"brief(ing)?|pre[- ]job\s*brief(ing)?|"
     r"campaign(s)?|coaching|"
     r"onboarding|orientation"
     r")\b"),

    # 5) Communication / Coordination  (handover, SIMOPS, interface, coordination)
    ("Communication/Coordination",
     r"\b("
     r"handover|hand[- ]over|shift[- ]handover|"
     r"communication(s)?|"
     r"coordination|coordinate|"
     r"interface|alignment|"
     r"radio|"
     r"simops|"
     r"contractor[- ]operations|joint\s*contractor|"
     r"brief(ing)?"
     r")\b"),

    # 6) Equipment / Engineering Change  (hardware + layout/design + systems/tools/tracking)
    ("Equipment/Engineering Change",
     r"\b("
     r"install|add|modify|upgrade|retrofit|replace|refurbish|"
     r"implement|configure|"
     r"equipment|hardware|device(s)?|"
     r"valve(s)?|check\s*valve(s)?|actuator(s)?|"
     r"bleed\s*point(s)?|vent\s*plug(s)?|drain(s)?|"
     r"filter(s)?|screen(s)?|anemometer(s)?|"
     r"alarm(s)?|control\s*system(s)?|"
     r"p\&id(s)?|p&id(s)?|drawing(s)?|diagram(s)?|schematic(s)?|isometric(s)?|"
     r"layout(s)?|redesign|reposition|re[- ]design|"
     r"tracking\s*system|centralized\s*tracking|automated\s*alert|"
     r"digital\s*ptw|ptw\s*system|"
     r"single[- ]source[- ]of[- ]truth|synchronize(d)?|synchroni[sz]e(d)?|"
     r"consolidate|central(ized)?"
     r")\b"),
]

In [ ]:
# ==========================================================
# 2) Classify + apply
# ==========================================================

def classify_action_type(action_title):
    t = _norm_text(action_title)

    if t == "":
        return "Unknown"

    for cat, pat in ACTION_TYPE_RULES:
        if re.search(pat, t, flags=re.IGNORECASE):
            return cat

    return "Other"


# If your column is action_text (renamed), use that; otherwise action_title
action_col = "action_title"
if "action_text" in df.columns and "action_title" not in df.columns:
    action_col = "action_text"

df["action_type"] = df[action_col].apply(classify_action_type)

df["action_type"].value_counts()

,count
action_type,
Audit/Verification/Assessment,504
Equipment/Engineering Change,323
Other,247
Procedure/Standard,235
Training/Drills,175
Visual Management/Signage,102
Communication/Coordination,17


In [ ]:
# Save
df.to_csv("actions_cleaned_v5.csv", index=False)
print("Saved: actions_cleaned_v5.csv")

Saved: actions_cleaned_v5.csv
